# Metadata Extraction from XML

Parses GROBID-generated TEI XML files and extracts structured metadata: title, authors, abstract, year, and full body text. Author names are validated to filter out common GROBID extraction errors (encoding artifacts, footnote fragments, etc.).

**Input:** `output/xml_english/` — TEI XML files produced by `PDFtoXML.ipynb`  
**Output:** `output/metadata_from_xml.csv`

In [ ]:
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd
from pydantic import BaseModel, Field

In [ ]:
PROJECT_ROOT = Path("..").resolve()

XML_DIR    = PROJECT_ROOT / "output" / "xml_english"  # TEI XML files from GROBID
OUTPUT_CSV = PROJECT_ROOT / "output" / "metadata_from_xml.csv"

MAX_AUTHORS = 15  # cap per paper to avoid parsing runaway reference lists

In [ ]:
class Author(BaseModel):
    surname: Optional[str] = None
    forename: Optional[str] = None

    @property
    def full_name(self) -> Optional[str]:
        if self.surname and self.forename:
            return f"{self.forename} {self.surname}"
        return self.surname or self.forename


class PaperMetadata(BaseModel):
    xml_file: str = Field(..., description="Source XML filename")
    pdf_file: str = Field(..., description="Corresponding PDF filename")
    title: Optional[str] = Field(None)
    authors: List[Author] = Field(default_factory=list)
    abstract: Optional[str] = Field(None)
    year: Optional[int] = Field(None)
    full_text: Optional[str] = Field(None)

In [ ]:
def is_likely_error(surname: Optional[str], forename: Optional[str] = None) -> Tuple[bool, Optional[str]]:
    """
    Heuristically detect malformed author names produced by GROBID.
    Covers digits, bad symbols, encoding errors, abbreviations, and implausible lengths.

    Returns (is_error, reason_code).
    """
    if not surname:
        return True, "empty_surname"

    if re.search(r"\d", surname):
        return True, "has_numbers"
    if forename and re.search(r"\d", forename):
        return True, "has_numbers_in_forename"

    bad_symbols = r"[{}~*•¢¡¶©_\[\]<>|\\^`´€£¥$]"
    if re.search(bad_symbols, surname):
        return True, "bad_symbols"
    if forename and re.search(bad_symbols, forename):
        return True, "bad_symbols_in_forename"

    if "\ufffd" in surname or (forename and "\ufffd" in forename):
        return True, "encoding_error"

    if re.search(r"[^A-Za-zÀ-ž]{3,}", surname):
        return True, "too_many_special"

    if re.match(r"^[^A-Za-zÀ-ž]", surname) or re.search(r"[^A-Za-zÀ-ž]$", surname):
        return True, "malformed"

    if " " in surname and surname.isupper() and len(surname.split()) > 2:
        return True, "uppercase_multi"

    if surname in {'"', "'", "-", "•", ".", "–", "—", ",", ";", ":", "!", "?"}:
        return True, "just_punctuation"
    if forename and forename in {'"', "'", "-", "•", ".", "–", "—", ",", ";", ":", "!", "?"}:
        return True, "punctuation_as_forename"

    if forename and forename.lower() in {"mr", "dr", "ms", "mrs", "prof", "sir", "dame"}:
        return True, "title_as_forename"

    if surname.lower() in {"gm", "im", "fm", "wgm", "wim", "cm", "fide"}:
        return True, "chess_title_as_surname"

    if surname.startswith("-"):
        return True, "starts_with_hyphen"

    if len(surname) == 1 and surname not in {"O", "Ó", "Ö", "Ø", "Ū"}:
        return True, "single_char"

    if re.match(r"^[A-Z]-\d+", surname):
        return True, "postal_code_pattern"

    if len(surname) <= 3 and surname.isupper() and forename and len(forename) > 2:
        return True, "likely_abbreviation"

    if len(surname) < 2 or len(surname) > 40:
        return True, "implausible_length"

    return False, None


def extract_metadata_from_xml(xml_file: Path) -> PaperMetadata:
    """
    Extract title, authors, abstract, year, and full text from a single TEI XML file.
    Returns a minimal PaperMetadata object on parse failure.
    """
    try:
        root = ET.parse(xml_file).getroot()
        ns = {"tei": "http://www.tei-c.org/ns/1.0"}

        pdf_filename = xml_file.stem.replace(".grobid.tei", "") + ".pdf"

        # Title
        title = None
        title_elem = root.find('.//tei:title[@level="a"][@type="main"]', ns)
        if title_elem is not None:
            title = "".join(title_elem.itertext()).strip() or None

        # Authors — scoped to analytic/sourceDesc to avoid picking up reference authors
        authors: List[Author] = []
        seen: set = set()
        for persName in root.findall(
            ".//tei:fileDesc/tei:sourceDesc/tei:biblStruct/tei:analytic/tei:author/tei:persName", ns
        ):
            surname_elem  = persName.find("tei:surname", ns)
            forename_elem = persName.find('tei:forename[@type="first"]', ns)

            surname  = surname_elem.text.strip()  if surname_elem  is not None and surname_elem.text  else None
            forename = forename_elem.text.strip() if forename_elem is not None and forename_elem.text else None

            is_error, _ = is_likely_error(surname, forename)
            if is_error:
                continue

            full_name = f"{forename} {surname}".strip() if forename else surname
            if full_name not in seen:
                authors.append(Author(surname=surname, forename=forename))
                seen.add(full_name)

        authors = authors[:MAX_AUTHORS]

        # Abstract
        abstract = None
        abstract_elem = root.find(".//tei:abstract", ns)
        if abstract_elem is not None:
            raw = " ".join("".join(abstract_elem.itertext()).split())
            abstract = raw or None

        # Full text from body
        full_text = None
        body_elem = root.find(".//tei:text/tei:body", ns)
        if body_elem is not None:
            raw = " ".join("".join(body_elem.itertext()).split())
            full_text = raw or None

        # Year
        year = None
        date_elem = root.find('.//tei:publicationStmt//tei:date[@type="published"]', ns)
        if date_elem is not None and date_elem.get("when"):
            try:
                year = int(date_elem.get("when")[:4])
            except (ValueError, TypeError):
                pass

        return PaperMetadata(
            xml_file=xml_file.name, pdf_file=pdf_filename,
            title=title, authors=authors, abstract=abstract,
            year=year, full_text=full_text,
        )

    except Exception:
        return PaperMetadata(
            xml_file=xml_file.name,
            pdf_file=xml_file.stem.replace(".grobid.tei", "") + ".pdf",
        )

In [ ]:
def create_metadata_dataset(xml_folder: Path, output_csv: Path) -> pd.DataFrame:
    """
    Extract metadata from all TEI XML files in xml_folder and save to output_csv.

    Returns a DataFrame with columns: xml_file, pdf_file, title, authors, abstract, year, full_text.
    """
    xml_files = sorted(xml_folder.glob("*.tei.xml"))
    print(f"Found {len(xml_files)} XML files in {xml_folder}\n")

    rows = []
    errors = no_authors = no_text = 0

    for i, f in enumerate(xml_files, 1):
        try:
            meta = extract_metadata_from_xml(f)
            rows.append({
                "xml_file":  meta.xml_file,
                "pdf_file":  meta.pdf_file,
                "title":     meta.title,
                "authors":   "; ".join(a.full_name for a in meta.authors if a.full_name),
                "abstract":  meta.abstract,
                "year":      meta.year,
                "full_text": meta.full_text,
            })
            if not meta.authors:   no_authors += 1
            if not meta.full_text: no_text    += 1
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  Error in {f.name}: {e}")

        if i % 100 == 0:
            print(f"  {i}/{len(xml_files)} processed...")

    df = pd.DataFrame(rows)
    output_csv.parent.mkdir(exist_ok=True, parents=True)
    df.to_csv(output_csv, index=False, encoding="utf-8")

    n = len(df)
    pct = lambda x: f"{x * 100 // n if n else 0}%"
    print(f"\nSaved {n} records -> {output_csv}")
    print(f"  With title:     {df['title'].notna().sum()} ({pct(df['title'].notna().sum())})")
    print(f"  With authors:   {n - no_authors} ({pct(n - no_authors)})")
    print(f"  With abstract:  {df['abstract'].notna().sum()} ({pct(df['abstract'].notna().sum())})")
    print(f"  With year:      {df['year'].notna().sum()} ({pct(df['year'].notna().sum())})")
    print(f"  With full text: {df['full_text'].notna().sum()} ({pct(df['full_text'].notna().sum())})")
    print(f"  Parse errors:   {errors}")
    return df

In [ ]:
df = create_metadata_dataset(XML_DIR, OUTPUT_CSV)

## Diagnostics (optional)

Run the cell below to inspect GROBID extraction quality — which author names were rejected and why.

In [ ]:
from collections import defaultdict


def _extract_authors_raw(xml_path: Path) -> List[dict]:
    """Return raw author dicts from a TEI XML file without any filtering."""
    try:
        root = ET.parse(xml_path).getroot()
        ns = {"tei": "http://www.tei-c.org/ns/1.0"}
        authors = []
        for author in root.findall(".//tei:sourceDesc//tei:author", ns):
            persName = author.find(".//tei:persName", ns)
            if persName is not None:
                s = persName.find(".//tei:surname", ns)
                f = persName.find(".//tei:forename", ns)
                authors.append({
                    "surname":  s.text.strip() if s is not None and s.text else None,
                    "forename": f.text.strip() if f is not None and f.text else None,
                })
        return authors
    except Exception:
        return []


def analyze_extraction_errors(xml_dir: Path, n: Optional[int] = None) -> None:
    """Print a breakdown of author extraction errors found in the XML files."""
    xml_files = sorted(xml_dir.glob("*.tei.xml"))
    if n:
        xml_files = xml_files[:n]

    print(f"Analysing {len(xml_files)} files...\n")

    real_errors = []
    files_without_authors = []

    for xml_file in xml_files:
        authors = _extract_authors_raw(xml_file)
        if not authors:
            files_without_authors.append(xml_file.name)
            continue
        for a in authors:
            is_error, reason = is_likely_error(a["surname"], a["forename"])
            if is_error:
                real_errors.append({"type": reason, **a, "file": xml_file.name})

    print(f"Files without authors: {len(files_without_authors)}")
    print(f"Extraction errors:     {len(real_errors)}\n")

    errors_by_type = defaultdict(list)
    for e in real_errors:
        errors_by_type[e["type"]].append(e)

    for error_type, examples in errors_by_type.items():
        print(f"{error_type.upper()} ({len(examples)}):")
        for e in examples[:5]:  # show at most 5 examples per type
            print(f"  {e['file']}: surname='{e['surname']}' forename='{e['forename']}'")


analyze_extraction_errors(XML_DIR)